In [ ]:
import os
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark is running!")

In [ ]:
# Load both CSV files
economy_df = spark.read.csv("../data/economy.csv", header=True, inferSchema=True)
business_df = spark.read.csv("../data/business.csv", header=True, inferSchema=True)

# Add class column to each
from pyspark.sql.functions import lit

economy_df = economy_df.withColumn("class", lit("Economy"))
business_df = business_df.withColumn("class", lit("Business"))

# Merge both
df = economy_df.union(business_df)

print("Total Rows:", df.count())
print("Columns:", df.columns)
df.show(3)

In [ ]:
# Schema check - datatypes dekho
df.printSchema()

In [ ]:
# Null values check karo har column mein
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])
null_counts.show()

In [ ]:
# Basic stats
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))
print("\nUnique Airlines:")
df.select("airline").distinct().show()
print("Unique Routes (from → to):")
df.select("from", "to").distinct().show()

In [ ]:
# inferSchema=False - sab string mein load karo
df_raw = spark.read.csv(
    "../data/economy.csv", 
    header=True, 
    inferSchema=False
)
df_raw.show(5, truncate=False)

In [ ]:
# Columns print karo
print(df_raw.columns)
# Ek row detail mein dekho
df_raw.limit(1).toPandas()

In [ ]:
from pyspark.sql.functions import lit

# inferSchema=False - apni marzi se types assign karenge hum
economy_df = spark.read.csv("../data/economy.csv", header=True, inferSchema=False)
business_df = spark.read.csv("../data/business.csv", header=True, inferSchema=False)

# Class column add karo
economy_df = economy_df.withColumn("class", lit("Economy"))
business_df = business_df.withColumn("class", lit("Business"))

# Dono merge karo
df = economy_df.union(business_df)

print("Total rows:", df.count())
df.show(3, truncate=False)

In [ ]:
from pyspark.sql.functions import regexp_replace, col

# price column clean karo
df = df.withColumn(
    "price",
    regexp_replace(col("price"), ",", "").cast("integer")
)

# Verify karo
df.select("price").show(5)

In [ ]:
from pyspark.sql.functions import regexp_extract, expr

# "02h 10m" se hours aur minutes alag nikalo
df = df.withColumn(
    "duration_minutes",
    (regexp_extract(col("time_taken"), r"(\d+)h", 1).cast("integer") * 60 +
     regexp_extract(col("time_taken"), r"(\d+)m", 1).cast("integer"))
)

# Verify karo
df.select("time_taken", "duration_minutes").show(5)

In [ ]:
from pyspark.sql.functions import when, substring

# dep_time se sirf hour nikalo
# "18:55" → "18" → integer
df = df.withColumn(
    "dep_hour",
    substring(col("dep_time"), 1, 2).cast("integer")
)

# Hour ke basis pe time of day assign karo
df = df.withColumn(
    "dep_time_of_day",
    when((col("dep_hour") >= 5) & (col("dep_hour") < 12), "Morning")
    .when((col("dep_hour") >= 12) & (col("dep_hour") < 17), "Afternoon")
    .when((col("dep_hour") >= 17) & (col("dep_hour") < 21), "Evening")
    .otherwise("Night")
)

# Verify karo
df.select("dep_time", "dep_hour", "dep_time_of_day").show(8)

In [ ]:
from pyspark.sql.functions import trim

# Extra spaces hatao aur standardize karo
df = df.withColumn(
    "stop",
    trim(col("stop"))
)

# Unique values dekho
df.select("stop").distinct().show()

In [ ]:
from pyspark.sql.functions import count, when, isnan

# Pehle count karo kitne NULLs hain
df.filter(col("stop").isNull()).count()

In [ ]:
# Dekho NULL stop wali rows kaisi dikhti hain
df.filter(col("stop").isNull()).show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import rlike

df = df.filter(
    col("date").rlike(r"^\d{2}-\d{2}-\d{4}$")
)

print("Rows after cleaning:", df.count())

In [ ]:
import os
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21"

from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, regexp_replace, col, regexp_extract, substring, when, trim
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("FlightAnalysis") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# Load
economy_df = spark.read.csv("../data/economy.csv", header=True, inferSchema=False)
business_df = spark.read.csv("../data/business.csv", header=True, inferSchema=False)

# Class column add
economy_df = economy_df.withColumn("class", lit("Economy"))
business_df = business_df.withColumn("class", lit("Business"))

# Merge
df = economy_df.union(business_df)

# Garbage rows filter
df = df.filter(col("date").rlike(r"^\d{2}-\d{2}-\d{4}$"))

# Price clean - try_cast use karo
df = df.withColumn("price",
    F.expr("try_cast(regexp_replace(price, ',', '') as int)")
)

# Duration clean - try_cast use karo
df = df.withColumn("hours",
    F.expr("try_cast(regexp_extract(time_taken, '(\\\\d+)h', 1) as int)")
)
df = df.withColumn("mins",
    F.expr("try_cast(regexp_extract(time_taken, '(\\\\d+)m', 1) as int)")
)
df = df.withColumn("duration_minutes",
    when(col("hours").isNull() | col("mins").isNull(), None)
    .otherwise(col("hours") * 60 + col("mins"))
)
df = df.drop("hours", "mins")

# Stop trim
df = df.withColumn("stop", trim(col("stop")))

# Dep hour
df = df.withColumn("dep_hour",
    F.expr("try_cast(substring(dep_time, 1, 2) as int)")
)

# Time of day
df = df.withColumn("dep_time_of_day",
    when((col("dep_hour") >= 5) & (col("dep_hour") < 12), "Morning")
    .when((col("dep_hour") >= 12) & (col("dep_hour") < 17), "Afternoon")
    .when((col("dep_hour") >= 17) & (col("dep_hour") < 21), "Evening")
    .otherwise("Night"))

# NULL rows drop karo
df = df.dropna(subset=["price", "duration_minutes", "airline", "from", "to"])

print("Clean rows:", df.count())
df.show(3, truncate=False)

In [ ]:
# PostgreSQL mein Dimension tables fill karo

import psycopg2

# PostgreSQL se connect karo
conn = psycopg2.connect(
    host="localhost",
    database="flight_analysis",
    user="postgres",
    password="postgres"  
)
cursor = conn.cursor()

# 1. dim_airline fill karo
airlines = df.select("airline").distinct().collect()
for row in airlines:
    cursor.execute(
        "INSERT INTO dim_airline (airline_name) VALUES (%s) ON CONFLICT DO NOTHING",
        (row["airline"],)
    )

# 2. dim_city fill karo
cities = df.select("from").distinct().collect()
for row in cities:
    cursor.execute(
        "INSERT INTO dim_city (city_name) VALUES (%s) ON CONFLICT DO NOTHING",
        (row["from"],)
    )

# 3. dim_time fill karo
times = ["Morning", "Afternoon", "Evening", "Night"]
for t in times:
    cursor.execute(
        "INSERT INTO dim_time (time_label) VALUES (%s) ON CONFLICT DO NOTHING",
        (t,)
    )

# 4. dim_class fill karo
classes = ["Economy", "Business"]
for c in classes:
    cursor.execute(
        "INSERT INTO dim_class (class_name) VALUES (%s) ON CONFLICT DO NOTHING",
        (c,)
    )

conn.commit()
print("Dimension tables filled! ✅")
cursor.close()
conn.close()

In [ ]:
import psycopg2
from pyspark.sql.functions import broadcast

# Connect karo
conn = psycopg2.connect(
    host="localhost",
    database="flight_analysis",
    user="postgres",
    password="postgres"
)
cursor = conn.cursor()

# Dimension tables se mappings load karo (ID lookup ke liye)
cursor.execute("SELECT airline_name, airline_id FROM dim_airline")
airline_map = {row[0]: row[1] for row in cursor.fetchall()}

cursor.execute("SELECT city_name, city_id FROM dim_city")
city_map = {row[0]: row[1] for row in cursor.fetchall()}

cursor.execute("SELECT time_label, time_id FROM dim_time")
time_map = {row[0]: row[1] for row in cursor.fetchall()}

cursor.execute("SELECT class_name, class_id FROM dim_class")
class_map = {row[0]: row[1] for row in cursor.fetchall()}

print("Mappings loaded!")
print("Airlines:", airline_map)
print("Cities:", city_map)
print("Times:", time_map)
print("Classes:", class_map)

In [ ]:
# Fact table fill karo
conn = psycopg2.connect(
    host="localhost",
    database="flight_analysis",
    user="postgres",
    password="postgres"
)
cursor = conn.cursor()

# PySpark df ko Python list mein convert karo
rows = df.select(
    "airline", "from", "to", 
    "dep_time_of_day", "class", 
    "stop", "duration_minutes", "price"
).collect()

print(f"Total rows to insert: {len(rows)}")

# Batch insert - 1000 rows ek baar mein
batch = []
count = 0

for row in rows:
    batch.append((
        airline_map.get(row["airline"]),
        city_map.get(row["from"]),
        city_map.get(row["to"]),
        time_map.get(row["dep_time_of_day"]),
        class_map.get(row["class"]),
        row["stop"],
        row["duration_minutes"],
        row["price"]
    ))
    
    if len(batch) == 1000:
        cursor.executemany("""
            INSERT INTO fact_flights 
            (airline_id, source_city_id, destination_city_id, 
             time_id, class_id, stop, duration_minutes, price)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, batch)
        conn.commit()
        count += len(batch)
        batch = []
        print(f"Inserted: {count} rows...")

# Remaining rows
if batch:
    cursor.executemany("""
        INSERT INTO fact_flights 
        (airline_id, source_city_id, destination_city_id, 
         time_id, class_id, stop, duration_minutes, price)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, batch)
    conn.commit()
    count += len(batch)

print(f"\nDone! Total inserted: {count} rows ✅")
cursor.close()
conn.close()